In [1]:
import os

RAW_PATH = os.path.join("..", "dataset", "raw", "include")
PROCESSED_PATH = os.path.join("..", "dataset", "processed")

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("RAW PATH:")
print(os.path.abspath(RAW_PATH))

print("\nPROCESSED PATH:")
print(os.path.abspath(PROCESSED_PATH))

RAW PATH:
c:\Users\palak\SANKETSETU\dataset\raw\include

PROCESSED PATH:
c:\Users\palak\SANKETSETU\dataset\processed


In [2]:
import os
import pandas as pd

video_records = []

for part in sorted(os.listdir(RAW_PATH)):
    part_path = os.path.join(RAW_PATH, part)

    if not os.path.isdir(part_path):
        continue

    for category in sorted(os.listdir(part_path)):
        category_path = os.path.join(part_path, category)

        if not os.path.isdir(category_path):
            continue

        for sign in sorted(os.listdir(category_path)):
            sign_path = os.path.join(category_path, sign)

            if not os.path.isdir(sign_path):
                continue

            for video in os.listdir(sign_path):
                if video.lower().endswith((".mov", ".mp4", ".avi")):
                    video_records.append({
                        "Category": category,
                        "Sign": sign,
                        "Video": video,
                        "Path": os.path.join(sign_path, video)
                    })

video_df = pd.DataFrame(video_records)

print("✅ Total Videos Found:", len(video_df))
video_df.head(10)

✅ Total Videos Found: 4257


,Category,Sign,Video,Path
0,Adjectives,1. loud,MVI_5177.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
1,Adjectives,1. loud,MVI_5178.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
2,Adjectives,1. loud,MVI_5179.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
3,Adjectives,1. loud,MVI_5257.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
4,Adjectives,1. loud,MVI_5258.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
5,Adjectives,1. loud,MVI_5259.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
6,Adjectives,1. loud,MVI_5335.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
7,Adjectives,1. loud,MVI_5336.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
8,Adjectives,1. loud,MVI_5337.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...
9,Adjectives,1. loud,MVI_9289.MOV,..\dataset\raw\include\Adjectives_1of8\Adjecti...


In [3]:
# Create processed folder hierarchy

created = 0

for _, row in video_df.iterrows():

    category_folder = os.path.join(PROCESSED_PATH, row["Category"])
    sign_folder = os.path.join(category_folder, row["Sign"])

    if not os.path.exists(sign_folder):
        os.makedirs(sign_folder, exist_ok=True)
        created += 1

print("✅ Sign folders created:", created)
print("Processed dataset root:", PROCESSED_PATH)

✅ Sign folders created: 262
Processed dataset root: ..\dataset\processed


In [4]:
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm
import os

# MediaPipe setup
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

test_df = video_df.head(3)      # ONLY FIRST 3 VIDEOS

processed = 0

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    cap = cv2.VideoCapture(row["Path"])
    sequence = []

    while cap.isOpened():
        success, frame = cap.read()

        if not success:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        frame_landmarks = np.zeros(63)

        if results.multi_hand_landmarks:
            hand = results.multi_hand_landmarks[0]

            coords = []
            for point in hand.landmark:
                coords.extend([point.x, point.y, point.z])

            frame_landmarks = np.array(coords)

        sequence.append(frame_landmarks)

    cap.release()

    sequence = np.array(sequence)

    save_path = os.path.join(
        PROCESSED_PATH,
        row["Category"],
        row["Sign"],
        row["Video"].replace(".MOV", ".npy").replace(".mov", ".npy")
    )

    np.save(save_path, sequence)
    processed += 1

print(f"\n✅ Test preprocessing complete!")
print("Videos processed:", processed)

100%|██████████| 3/3 [00:16<00:00,  5.59s/it]


✅ Test preprocessing complete!
Videos processed: 3


In [5]:
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm
import os

# MediaPipe setup
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

processed = 0
skipped = 0

for _, row in tqdm(video_df.iterrows(), total=len(video_df), desc="Processing INCLUDE"):

    # Output file path
    save_path = os.path.join(
        PROCESSED_PATH,
        row["Category"],
        row["Sign"],
        os.path.splitext(row["Video"])[0] + ".npy"
    )

    # Skip if already processed
    if os.path.exists(save_path):
        skipped += 1
        continue

    cap = cv2.VideoCapture(row["Path"])
    sequence = []

    while cap.isOpened():
        success, frame = cap.read()

        if not success:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        frame_landmarks = np.zeros(63, dtype=np.float32)

        if results.multi_hand_landmarks:
            hand = results.multi_hand_landmarks[0]

            coords = []
            for point in hand.landmark:
                coords.extend([point.x, point.y, point.z])

            frame_landmarks = np.array(coords, dtype=np.float32)

        sequence.append(frame_landmarks)

    cap.release()

    sequence = np.array(sequence, dtype=np.float32)
    np.save(save_path, sequence)

    processed += 1

hands.close()

print("\n🎉 INCLUDE preprocessing finished!")
print("Videos processed :", processed)
print("Videos skipped   :", skipped)
print("Total videos     :", processed + skipped)

Processing INCLUDE:   8%|▊         | 320/4257 [1:10:58<14:33:14, 13.31s/it]


KeyboardInterrupt: 

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ==========================
# PATHS
# ==========================
RAW_PATH = os.path.join("..", "dataset", "raw", "include")
PROCESSED_PATH = os.path.join("..", "dataset", "processed")

# ==========================
# LANDMARK EXTRACTION
# ==========================
def extract_landmarks(video_path):
    mp_hands = mp.solutions.hands

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)
    sequence = []

    while cap.isOpened():
        success, frame = cap.read()

        if not success:
            break

        # Resize frame for faster inference
        frame = cv2.resize(frame, (320, 240))

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        landmarks = np.zeros(63, dtype=np.float32)

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]

            coords = []
            for lm in hand.landmark:
                coords.extend([lm.x, lm.y, lm.z])

            landmarks = np.array(coords, dtype=np.float32)

        sequence.append(landmarks)

    cap.release()
    hands.close()

    return np.array(sequence, dtype=np.float32)


# ==========================
# PROCESS ONE VIDEO
# ==========================
def process_video(task):
    category, sign, video_path, output_path = task

    # Skip if already processed
    if os.path.exists(output_path):
        return "skip"

    try:
        sequence = extract_landmarks(video_path)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        np.save(output_path, sequence)

        return "done"

    except Exception:
        # Ignore corrupted videos and continue
        return "error"


# ==========================
# CREATE TASK LIST
# ==========================
tasks = []

for part in sorted(os.listdir(RAW_PATH)):
    part_path = os.path.join(RAW_PATH, part)

    if not os.path.isdir(part_path):
        continue

    for category in os.listdir(part_path):
        category_path = os.path.join(part_path, category)

        if not os.path.isdir(category_path):
            continue

        for sign in os.listdir(category_path):
            sign_path = os.path.join(category_path, sign)

            if not os.path.isdir(sign_path):
                continue

            for video in os.listdir(sign_path):

                if not video.lower().endswith(".mov"):
                    continue

                video_path = os.path.join(sign_path, video)

                output_name = os.path.splitext(video)[0] + ".npy"

                output_path = os.path.join(
                    PROCESSED_PATH,
                    category,
                    sign,
                    output_name
                )

                tasks.append((category, sign, video_path, output_path))

print(f"📹 Remaining/Total task entries found: {len(tasks)}")

# ==========================
# MULTI-THREAD PROCESSING
# ==========================
workers = min(6, multiprocessing.cpu_count())

print(f"🚀 Using {workers} CPU threads")

done = 0
skipped = 0
errors = 0

with ThreadPoolExecutor(max_workers=workers) as executor:

    results = executor.map(process_video, tasks)

    for result in tqdm(results, total=len(tasks), desc="Processing INCLUDE"):

        if result == "done":
            done += 1

        elif result == "skip":
            skipped += 1

        elif result == "error":
            errors += 1


print("\n==============================")
print("✅ PREPROCESSING FINISHED")
print("==============================")
print(f"New videos processed : {done}")
print(f"Skipped existing     : {skipped}")
print(f"Errors               : {errors}")
print("==============================")

📹 Remaining/Total task entries found: 3652
🚀 Using 6 CPU threads


Processing INCLUDE:  29%|██▉       | 1071/3652 [42:58<2:54:12,  4.05s/it]

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ==========================
# PATHS
# ==========================
RAW_PATH = os.path.join("..", "dataset", "raw", "include")
PROCESSED_PATH = os.path.join("..", "dataset", "processed")

# ==========================
# LANDMARK EXTRACTION
# ==========================
def extract_landmarks(video_path):
    mp_hands = mp.solutions.hands

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)
    sequence = []

    while cap.isOpened():
        success, frame = cap.read()

        if not success:
            break

        # Resize frame for faster inference
        frame = cv2.resize(frame, (320, 240))

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        landmarks = np.zeros(63, dtype=np.float32)

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]

            coords = []
            for lm in hand.landmark:
                coords.extend([lm.x, lm.y, lm.z])

            landmarks = np.array(coords, dtype=np.float32)

        sequence.append(landmarks)

    cap.release()
    hands.close()

    return np.array(sequence, dtype=np.float32)


# ==========================
# PROCESS ONE VIDEO
# ==========================
def process_video(task):
    category, sign, video_path, output_path = task

    # Skip if already processed
    if os.path.exists(output_path):
        return "skip"

    try:
        sequence = extract_landmarks(video_path)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        np.save(output_path, sequence)

        return "done"

    except Exception:
        # Ignore corrupted videos and continue
        return "error"


# ==========================
# CREATE TASK LIST
# ==========================
tasks = []

for part in sorted(os.listdir(RAW_PATH)):
    part_path = os.path.join(RAW_PATH, part)

    if not os.path.isdir(part_path):
        continue

    for category in os.listdir(part_path):
        category_path = os.path.join(part_path, category)

        if not os.path.isdir(category_path):
            continue

        for sign in os.listdir(category_path):
            sign_path = os.path.join(category_path, sign)

            if not os.path.isdir(sign_path):
                continue

            for video in os.listdir(sign_path):

                if not video.lower().endswith(".mov"):
                    continue

                video_path = os.path.join(sign_path, video)

                output_name = os.path.splitext(video)[0] + ".npy"

                output_path = os.path.join(
                    PROCESSED_PATH,
                    category,
                    sign,
                    output_name
                )

                tasks.append((category, sign, video_path, output_path))

print(f"📹 Remaining/Total task entries found: {len(tasks)}")

# ==========================
# MULTI-THREAD PROCESSING
# ==========================
workers = min(6, multiprocessing.cpu_count())

print(f"🚀 Using {workers} CPU threads")

done = 0
skipped = 0
errors = 0

with ThreadPoolExecutor(max_workers=workers) as executor:

    results = executor.map(process_video, tasks)

    for result in tqdm(results, total=len(tasks), desc="Processing INCLUDE"):

        if result == "done":
            done += 1

        elif result == "skip":
            skipped += 1

        elif result == "error":
            errors += 1


print("\n==============================")
print("✅ PREPROCESSING FINISHED")
print("==============================")
print(f"New videos processed : {done}")
print(f"Skipped existing     : {skipped}")
print(f"Errors               : {errors}")
print("==============================")

📹 Remaining/Total task entries found: 3652
🚀 Using 6 CPU threads


Processing INCLUDE:  84%|████████▎ | 3053/3652 [1:25:53<17:39,  1.77s/it]  

In [1]:
import os
import cv2
import numpy as np
import mediapipe as mp
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ==========================
# PATHS
# ==========================
RAW_PATH = os.path.join("..", "dataset", "raw", "include")
PROCESSED_PATH = os.path.join("..", "dataset", "processed")

# ==========================
# LANDMARK EXTRACTION
# ==========================
def extract_landmarks(video_path):
    mp_hands = mp.solutions.hands

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)
    sequence = []

    while cap.isOpened():
        success, frame = cap.read()

        if not success:
            break

        # Resize frame for faster inference
        frame = cv2.resize(frame, (320, 240))

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        landmarks = np.zeros(63, dtype=np.float32)

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]

            coords = []
            for lm in hand.landmark:
                coords.extend([lm.x, lm.y, lm.z])

            landmarks = np.array(coords, dtype=np.float32)

        sequence.append(landmarks)

    cap.release()
    hands.close()

    return np.array(sequence, dtype=np.float32)


# ==========================
# PROCESS ONE VIDEO
# ==========================
def process_video(task):
    category, sign, video_path, output_path = task

    # Skip if already processed
    if os.path.exists(output_path):
        return "skip"

    try:
        sequence = extract_landmarks(video_path)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        np.save(output_path, sequence)

        return "done"

    except Exception:
        # Ignore corrupted videos and continue
        return "error"


# ==========================
# CREATE TASK LIST
# ==========================
tasks = []

for part in sorted(os.listdir(RAW_PATH)):
    part_path = os.path.join(RAW_PATH, part)

    if not os.path.isdir(part_path):
        continue

    for category in os.listdir(part_path):
        category_path = os.path.join(part_path, category)

        if not os.path.isdir(category_path):
            continue

        for sign in os.listdir(category_path):
            sign_path = os.path.join(category_path, sign)

            if not os.path.isdir(sign_path):
                continue

            for video in os.listdir(sign_path):

                if not video.lower().endswith(".mov"):
                    continue

                video_path = os.path.join(sign_path, video)

                output_name = os.path.splitext(video)[0] + ".npy"

                output_path = os.path.join(
                    PROCESSED_PATH,
                    category,
                    sign,
                    output_name
                )

                tasks.append((category, sign, video_path, output_path))

print(f"📹 Remaining/Total task entries found: {len(tasks)}")

# ==========================
# MULTI-THREAD PROCESSING
# ==========================
workers = min(6, multiprocessing.cpu_count())

print(f"🚀 Using {workers} CPU threads")

done = 0
skipped = 0
errors = 0

with ThreadPoolExecutor(max_workers=workers) as executor:

    results = executor.map(process_video, tasks)

    for result in tqdm(results, total=len(tasks), desc="Processing INCLUDE"):

        if result == "done":
            done += 1

        elif result == "skip":
            skipped += 1

        elif result == "error":
            errors += 1


print("\n==============================")
print("✅ PREPROCESSING FINISHED")
print("==============================")
print(f"New videos processed : {done}")
print(f"Skipped existing     : {skipped}")
print(f"Errors               : {errors}")
print("==============================")

📹 Remaining/Total task entries found: 3652
🚀 Using 6 CPU threads


Processing INCLUDE: 100%|██████████| 3652/3652 [48:19<00:00,  1.26it/s]   


✅ PREPROCESSING FINISHED
New videos processed : 597
Skipped existing     : 3055
Errors               : 0
